In [1]:

import os
import json
import math
import random
import warnings
from collections import defaultdict

import numpy as np
from scipy.integrate import trapezoid
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from scipy.stats import spearmanr, wilcoxon, friedmanchisquare
from sklearn.model_selection import train_test_split
from transformers import RobertaModel, RobertaTokenizer

warnings.filterwarnings("ignore")

# ============================================================
# CONFIG
# ============================================================

MODEL_PATH = "/home/btlab/Desktop/Fahim/LLM Misinfo/runs_emotion_cls_2/best_model_cls.pt"
DATA_PATH = "/home/btlab/Desktop/Fahim/LLM Misinfo/EmoBlend Fusion2_THMS/text.csv"
OUT_DIR = "/home/btlab/Desktop/Fahim/LLM Misinfo/runs_emotion_cls_2/xai_faithfulness_v6_pubready_figures"

TEST_SIZE = 0.10
SPLIT_RANDOM_STATE = 42
GLOBAL_SEED = 42

MAX_LENGTH = 64
M_STEPS_SCREEN = 16
M_STEPS_DEEP = 32

EVAL_POOL_SIZE = 600
N_RUNS = 7
RUN_SAMPLE_SIZE = 400

FRACTIONS = np.linspace(0.1, 1.0, 10)
TOPK_FRAC_COMP = 0.20
TOPK_FRAC_SWEEP = [0.05, 0.10, 0.15, 0.20, 0.30]
RANDOM_RESTARTS = 5

FIXED_ALPHA_SWEEP = [0.60, 0.70, 0.80, 0.85, 0.90, 0.95]
NORMALIZATION_SWEEP = ["sum", "minmax", "rank", "softmax"]
ADAPTIVE_ALPHA_RANGES = [
    (0.10, 0.60),
    (0.65, 0.95),
    (0.70, 0.95),
    (0.75, 0.95),
]

PRIMARY_BASELINES = ["attention", "ig", "random"]
BOOTSTRAP_B = 2000
BOOTSTRAP_SEED = 123
DPI = 180

os.makedirs(OUT_DIR, exist_ok=True)

# ============================================================
# REPRODUCIBILITY
# ============================================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(GLOBAL_SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f">>> Using device: {device}")

# ============================================================
# MODEL
# ============================================================

class ImprovedDeepEmotionModel(nn.Module):
    def __init__(self, num_classes=6, dropout=0.3):
        super().__init__()

        self.roberta = RobertaModel.from_pretrained("roberta-base")
        for p in self.roberta.parameters():
            p.requires_grad = False

        self.cnn = nn.Conv1d(
            in_channels=768,
            out_channels=128,
            kernel_size=3,
            padding=1
        )
        self.maxpool = nn.MaxPool1d(kernel_size=2, stride=2)

        self.bilstm = nn.LSTM(
            input_size=128,
            hidden_size=256,
            num_layers=1,
            bidirectional=True,
            batch_first=True,
        )

        self.attention = nn.MultiheadAttention(
            embed_dim=512,
            num_heads=8,
            dropout=0.1,
            batch_first=True
        )

        self.cls_query = nn.Parameter(torch.randn(1, 1, 512))

        self.dropout = nn.Dropout(dropout)
        self.layernorm = nn.LayerNorm(512)
        self.fc = nn.Linear(512, num_classes)

    def _mha_with_weights(self, query, key, value, key_padding_mask=None):
        try:
            attn_out, attn_weights = self.attention(
                query=query,
                key=key,
                value=value,
                key_padding_mask=key_padding_mask,
                need_weights=True,
                average_attn_weights=False
            )
        except TypeError:
            attn_out, attn_weights = self.attention(
                query=query,
                key=key,
                value=value,
                key_padding_mask=key_padding_mask,
                need_weights=True
            )
            if attn_weights.dim() == 3:
                attn_weights = attn_weights.unsqueeze(1)
        return attn_out, attn_weights

    def forward(self, input_ids, attention_mask, return_attn=False, return_embeddings=False):
        roberta_out = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        emb = roberta_out.last_hidden_state

        x = emb.transpose(1, 2)
        x = F.relu(self.cnn(x))
        x = self.maxpool(x)
        x = x.transpose(1, 2)

        pooled_mask = attention_mask[:, ::2]
        pooled_mask = pooled_mask[:, :x.size(1)]
        key_padding_mask = ~pooled_mask.bool()

        x, _ = self.bilstm(x)
        q = self.cls_query.expand(x.size(0), -1, -1)
        attn_out, attn_weights = self._mha_with_weights(
            q, x, x, key_padding_mask=key_padding_mask
        )

        pooled = attn_out.squeeze(1)
        pooled = self.layernorm(pooled)
        pooled = self.dropout(pooled)
        logits = self.fc(pooled)

        out = {"logits": logits}
        if return_attn:
            out["attn_weights"] = attn_weights
            out["pooled_mask"] = pooled_mask
        if return_embeddings:
            out["embeddings"] = emb
            out["emb_mask"] = attention_mask.bool()
        return out


def forward_from_embeddings(model, emb, emb_mask):
    x = emb.transpose(1, 2)
    x = F.relu(model.cnn(x))
    x = model.maxpool(x)
    x = x.transpose(1, 2)

    pooled_mask = emb_mask[:, ::2]
    pooled_mask = pooled_mask[:, :x.size(1)]
    key_padding_mask = ~pooled_mask.bool()

    x, _ = model.bilstm(x)
    q = model.cls_query.expand(x.size(0), -1, -1)
    attn_out, _ = model._mha_with_weights(q, x, x, key_padding_mask=key_padding_mask)

    pooled = attn_out.squeeze(1)
    pooled = model.layernorm(pooled)
    pooled = model.dropout(pooled)
    logits = model.fc(pooled)
    return logits

# ============================================================
# HELPERS
# ============================================================

def safe_spearman(a, b):
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    if len(a) < 2 or len(b) < 2:
        return 0.0
    if np.allclose(a, a[0]) or np.allclose(b, b[0]):
        return 0.0
    corr, _ = spearmanr(a, b)
    if corr is None or np.isnan(corr):
        return 0.0
    return float(corr)


def entropy_normalized(x, eps=1e-12):
    x = np.asarray(x, dtype=np.float64)
    x = np.clip(x, a_min=0.0, a_max=None)
    s = x.sum()
    if s <= eps:
        return 1.0
    p = x / s
    h = -(p * np.log(p + eps)).sum()
    hmax = np.log(len(p) + eps)
    if hmax <= eps:
        return 1.0
    return float(h / hmax)


def normalize_with_mask(scores, valid_mask, mode="sum", eps=1e-12):
    scores = np.asarray(scores, dtype=np.float64)
    valid_mask = np.asarray(valid_mask, dtype=bool)
    out = np.zeros_like(scores, dtype=np.float64)

    if valid_mask.sum() == 0:
        return out

    vals = np.asarray(scores[valid_mask], dtype=np.float64)
    vals = np.abs(vals)

    if mode == "sum":
        vals = np.clip(vals, a_min=0.0, a_max=None)
    elif mode == "minmax":
        vmin = vals.min()
        vmax = vals.max()
        if vmax - vmin <= eps:
            vals = np.ones_like(vals, dtype=np.float64)
        else:
            vals = (vals - vmin) / (vmax - vmin)
    elif mode == "rank":
        order = np.argsort(vals)
        ranks = np.empty_like(order, dtype=np.float64)
        ranks[order] = np.arange(1, len(vals) + 1, dtype=np.float64)
        vals = ranks
    elif mode == "softmax":
        shifted = vals - vals.max()
        vals = np.exp(shifted)
    else:
        raise ValueError(f"Unknown normalization mode: {mode}")

    vals = np.clip(vals, a_min=0.0, a_max=None)
    s = vals.sum()
    if s <= eps:
        vals = np.ones_like(vals, dtype=np.float64) / len(vals)
    else:
        vals = vals / s

    out[valid_mask] = vals
    return out


def bootstrap_mean_ci(x, n_boot=2000, seed=123, ci=95):
    x = np.asarray(x, dtype=np.float64)
    rng = np.random.default_rng(seed)
    means = []
    n = len(x)
    for _ in range(n_boot):
        sample = rng.choice(x, size=n, replace=True)
        means.append(sample.mean())
    low = np.percentile(means, (100 - ci) / 2)
    high = np.percentile(means, 100 - (100 - ci) / 2)
    return float(low), float(high)


def holm_bonferroni(df, p_col="p_value"):
    df = df.copy().sort_values(p_col).reset_index(drop=True)
    m = len(df)
    adjusted = []
    running_max = 0.0
    for i, p in enumerate(df[p_col].tolist()):
        adj = (m - i) * p
        running_max = max(running_max, adj)
        adjusted.append(min(running_max, 1.0))
    df["p_holm"] = adjusted
    return df


def stratified_subsample_indices(y, max_n, seed):
    y = np.asarray(y)
    idx = np.arange(len(y))
    if max_n is None or max_n >= len(y):
        return idx
    chosen, _ = train_test_split(
        idx,
        train_size=max_n,
        stratify=y,
        random_state=seed
    )
    return np.sort(chosen)


def build_deletable_mask(input_ids_1d, attention_mask_1d, tokenizer):
    mask = attention_mask_1d.bool().clone()
    special_ids = [
        tokenizer.pad_token_id,
        tokenizer.cls_token_id,
        tokenizer.sep_token_id,
    ]
    for sid in special_ids:
        if sid is not None:
            mask &= (input_ids_1d != sid)
    return mask


def adaptive_alpha_from_scores(attn_scores, ig_scores, alpha_min=0.10, alpha_max=0.60):
    corr = safe_spearman(attn_scores, ig_scores)
    corr01 = 0.5 * (corr + 1.0)
    attn_ent = entropy_normalized(attn_scores)
    attn_concentration = 1.0 - attn_ent

    raw = 0.60 * corr01 + 0.40 * attn_concentration
    alpha = alpha_min + (alpha_max - alpha_min) * raw
    alpha = float(np.clip(alpha, alpha_min, alpha_max))
    return alpha, corr, attn_ent


def method_tag(method_type, **kwargs):
    if method_type in ["attention", "ig", "random"]:
        return method_type
    if method_type == "hybrid_fixed":
        return f"hybrid_fixed__norm={kwargs['norm']}__alpha={kwargs['alpha']:.2f}"
    if method_type == "hybrid_adaptive":
        return (
            f"hybrid_adaptive__norm={kwargs['norm']}"
            f"__amin={kwargs['alpha_min']:.2f}__amax={kwargs['alpha_max']:.2f}"
        )
    raise ValueError(method_type)


def parse_method_tag(tag):
    if tag in ["attention", "ig", "random"]:
        return {"method_type": tag}
    parts = tag.split("__")
    method_type = parts[0]
    out = {"method_type": method_type}
    for p in parts[1:]:
        k, v = p.split("=")
        if k == "norm":
            out[k] = v
        else:
            out[k] = float(v)
    return out


def make_candidate_specs():
    specs = []
    for base in PRIMARY_BASELINES:
        specs.append({"name": base, "method_type": base})

    for norm in NORMALIZATION_SWEEP:
        for alpha in FIXED_ALPHA_SWEEP:
            specs.append({
                "name": method_tag("hybrid_fixed", norm=norm, alpha=alpha),
                "method_type": "hybrid_fixed",
                "norm": norm,
                "alpha": alpha,
            })
        for alpha_min, alpha_max in ADAPTIVE_ALPHA_RANGES:
            specs.append({
                "name": method_tag(
                    "hybrid_adaptive",
                    norm=norm,
                    alpha_min=alpha_min,
                    alpha_max=alpha_max,
                ),
                "method_type": "hybrid_adaptive",
                "norm": norm,
                "alpha_min": alpha_min,
                "alpha_max": alpha_max,
            })
    return specs


CANDIDATE_SPECS = make_candidate_specs()
CANDIDATE_NAMES = [spec["name"] for spec in CANDIDATE_SPECS]
print(f">>> Candidate methods in screening stage: {len(CANDIDATE_NAMES)}")


def set_dropout_eval_only(model):
    model.dropout.eval()


def restore_modes(model, old_modes):
    for module, mode in old_modes:
        module.train(mode)


def prepare_model_for_ig(model):
    old_modes = [
        (model, model.training),
        (model.bilstm, model.bilstm.training),
        (model.dropout, model.dropout.training),
        (model.attention, model.attention.training),
        (model.layernorm, model.layernorm.training),
        (model.fc, model.fc.training),
    ]

    model.train()
    model.bilstm.train()
    model.attention.train()
    model.fc.train()
    model.layernorm.train()
    model.dropout.eval()
    return old_modes

# ============================================================
# DATA
# ============================================================

print(">>> Loading data...")
df = pd.read_csv(DATA_PATH)
if "text" not in df.columns or "label" not in df.columns:
    raise ValueError("CSV must contain 'text' and 'label' columns.")

texts = df["text"].astype(str).tolist()
labels = df["label"].astype(int).tolist()
num_classes = len(sorted(set(labels)))

tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

X_train, X_test, y_train, y_test = train_test_split(
    texts,
    labels,
    test_size=TEST_SIZE,
    stratify=labels,
    random_state=SPLIT_RANDOM_STATE,
)

test_df = pd.DataFrame({"text": X_test, "label": y_test})
if EVAL_POOL_SIZE is not None and EVAL_POOL_SIZE < len(test_df):
    keep_idx = stratified_subsample_indices(test_df["label"].values, EVAL_POOL_SIZE, seed=GLOBAL_SEED)
    test_df = test_df.iloc[keep_idx].reset_index(drop=True)

print(f">>> Test instances available for XAI analysis: {len(test_df)}")

# ============================================================
# LOAD MODEL
# ============================================================

print(">>> Loading model...")
model = ImprovedDeepEmotionModel(num_classes=num_classes, dropout=0.3).to(device)
state = torch.load(MODEL_PATH, map_location=device)

missing, unexpected = model.load_state_dict(state, strict=False)
print(">>> Missing keys:", missing)
print(">>> Unexpected keys:", unexpected)

if len(missing) > 0 or len(unexpected) > 0:
    print(">>> Warning: state_dict did not load perfectly. Please inspect keys above.")

model.eval()
print(">>> Model loaded successfully.")

# ============================================================
# PREDICTION HELPERS
# ============================================================

def predict_prob_for_class(model, input_ids_1d, class_idx, pad_id):
    if input_ids_1d.dim() == 1:
        input_ids = input_ids_1d.unsqueeze(0).to(device)
    else:
        input_ids = input_ids_1d.to(device)
    attention_mask = (input_ids != pad_id).long()

    with torch.no_grad():
        logits = model(input_ids, attention_mask)["logits"]
        probs = F.softmax(logits, dim=-1)
    return float(probs[0, class_idx].item())


def compute_attention_token_scores(out):
    attn_weights = out["attn_weights"]
    pooled_mask = out["pooled_mask"][0]
    emb_mask = out["emb_mask"][0]

    attn = attn_weights[0]
    if attn.dim() == 2:
        A_pooled = attn.squeeze(0)
    elif attn.dim() == 3:
        A_pooled = attn.mean(dim=0).squeeze(0)
    else:
        raise ValueError(f"Unexpected attention weight shape: {attn.shape}")

    A_pooled = A_pooled * pooled_mask.float()
    A_pooled = A_pooled / (A_pooled.sum() + 1e-8)

    T = emb_mask.size(0)
    S = A_pooled.size(0)
    A_tokens = torch.zeros(T, device=device)

    for i in range(S):
        if not pooled_mask[i]:
            continue
        t1 = 2 * i
        t2 = 2 * i + 1
        if t1 < T:
            A_tokens[t1] += A_pooled[i]
        if t2 < T:
            A_tokens[t2] += A_pooled[i]

    return A_tokens.detach().cpu().numpy()


def compute_ig_token_scores(model, embeddings, emb_mask, pred_class, m_steps=16):
    baseline = torch.zeros_like(embeddings).to(device)
    scaled_embs = []
    for k in range(1, m_steps + 1):
        alpha_k = float(k) / m_steps
        scaled = baseline + alpha_k * (embeddings - baseline)
        scaled_embs.append(scaled)

    scaled_embs = torch.cat(scaled_embs, dim=0)
    scaled_masks = emb_mask.unsqueeze(0).repeat(m_steps, 1).to(device)

    scaled_embs.requires_grad_(True)
    grads_accum = torch.zeros_like(scaled_embs)

    old_modes = prepare_model_for_ig(model)
    try:
        with torch.backends.cudnn.flags(enabled=False):
            for i in range(m_steps):
                emb_i = scaled_embs[i:i+1]
                mask_i = scaled_masks[i:i+1]
                logits_i = forward_from_embeddings(model, emb_i, mask_i)
                logit_c = logits_i[0, pred_class]
                grads = torch.autograd.grad(
                    logit_c,
                    emb_i,
                    retain_graph=True,
                    create_graph=False,
                    allow_unused=False,
                )[0]
                grads_accum[i] = grads[0]
    finally:
        restore_modes(model, old_modes)
        model.eval()

    avg_grads = grads_accum.mean(dim=0)
    ig = (embeddings[0] - baseline[0]) * avg_grads
    return ig.abs().sum(dim=-1).detach().cpu().numpy()


def explain_instance(model, tokenizer, text, true_label, m_steps=16, max_length=64):
    enc = tokenizer(
        text,
        padding="max_length",
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
    )
    input_ids = enc["input_ids"].to(device)
    attention_mask = enc["attention_mask"].to(device)

    with torch.no_grad():
        out = model(input_ids, attention_mask, return_attn=True, return_embeddings=True)

    logits = out["logits"]
    probs = F.softmax(logits, dim=-1)[0]
    pred_class = int(probs.argmax().item())
    pred_conf = float(probs[pred_class].item())

    embeddings = out["embeddings"]
    emb_mask = out["emb_mask"][0]

    input_ids_1d = input_ids[0].detach().cpu()
    attention_mask_1d = attention_mask[0].detach().cpu()
    deletable_mask_t = build_deletable_mask(input_ids_1d, attention_mask_1d, tokenizer)
    deletable_mask = deletable_mask_t.numpy().astype(bool)

    A_raw = compute_attention_token_scores(out)
    G_raw = compute_ig_token_scores(model, embeddings, emb_mask, pred_class, m_steps=m_steps)

    tokens = tokenizer.convert_ids_to_tokens(input_ids_1d.tolist())
    deletable_indices = np.where(deletable_mask)[0]
    visible_tokens = [tokens[i] for i in deletable_indices]

    return {
        "text": text,
        "true_label": int(true_label),
        "pred_label": pred_class,
        "pred_conf": pred_conf,
        "input_ids": input_ids_1d.clone(),
        "attention_mask": attention_mask_1d.clone(),
        "tokens_full": tokens,
        "visible_tokens": visible_tokens,
        "deletable_indices": deletable_indices,
        "deletable_mask": deletable_mask,
        "num_deletable": int(len(deletable_indices)),
        "attention_raw_scores": A_raw,
        "ig_raw_scores": G_raw,
    }

# ============================================================
# METHOD SCORE BUILDERS
# ============================================================

def build_score_bundle(record, norm="sum", alpha_fixed=0.85, alpha_min=0.65, alpha_max=0.95):
    deletable_mask = record["deletable_mask"]
    A_tokens = normalize_with_mask(record["attention_raw_scores"], deletable_mask, mode=norm)
    G_tokens = normalize_with_mask(record["ig_raw_scores"], deletable_mask, mode=norm)

    valid_attn = A_tokens[deletable_mask]
    valid_ig = G_tokens[deletable_mask]
    alpha_adaptive, corr_attn_ig, attn_entropy = adaptive_alpha_from_scores(
        valid_attn,
        valid_ig,
        alpha_min=alpha_min,
        alpha_max=alpha_max,
    )

    H_fixed = normalize_with_mask(
        alpha_fixed * A_tokens + (1.0 - alpha_fixed) * G_tokens,
        deletable_mask,
        mode="sum",
    )
    H_adaptive = normalize_with_mask(
        alpha_adaptive * A_tokens + (1.0 - alpha_adaptive) * G_tokens,
        deletable_mask,
        mode="sum",
    )

    return {
        "attention_scores": A_tokens,
        "ig_scores": G_tokens,
        "hybrid_fixed_scores": H_fixed,
        "hybrid_adaptive_scores": H_adaptive,
        "alpha_fixed": alpha_fixed,
        "alpha_adaptive": alpha_adaptive,
        "spearman_attn_ig": corr_attn_ig,
        "attn_entropy": attn_entropy,
        "norm": norm,
        "alpha_min": alpha_min,
        "alpha_max": alpha_max,
    }


def get_scores_for_method(record, method_name):
    parsed = parse_method_tag(method_name)
    mtype = parsed["method_type"]

    if mtype == "attention":
        bundle = build_score_bundle(record, norm="sum", alpha_fixed=0.85, alpha_min=0.65, alpha_max=0.95)
        return bundle["attention_scores"], bundle
    if mtype == "ig":
        bundle = build_score_bundle(record, norm="sum", alpha_fixed=0.85, alpha_min=0.65, alpha_max=0.95)
        return bundle["ig_scores"], bundle
    if mtype == "hybrid_fixed":
        bundle = build_score_bundle(
            record,
            norm=parsed["norm"],
            alpha_fixed=parsed["alpha"],
            alpha_min=0.65,
            alpha_max=0.95,
        )
        return bundle["hybrid_fixed_scores"], bundle
    if mtype == "hybrid_adaptive":
        bundle = build_score_bundle(
            record,
            norm=parsed["norm"],
            alpha_fixed=0.85,
            alpha_min=parsed["amin"],
            alpha_max=parsed["amax"],
        )
        return bundle["hybrid_adaptive_scores"], bundle
    if mtype == "random":
        return None, {}
    raise ValueError(method_name)

# ============================================================
# FAITHFULNESS METRICS
# ============================================================

def rank_positions(record, method_name, rng=None):
    deletable = record["deletable_indices"]
    if method_name == "random":
        if rng is None:
            rng = np.random.default_rng()
        order = deletable.copy()
        rng.shuffle(order)
        return order

    scores, _ = get_scores_for_method(record, method_name)
    order = deletable[np.argsort(-scores[deletable])]
    return order


def make_removed_input(input_ids_1d, remove_positions, pad_id):
    x = input_ids_1d.clone()
    x[list(remove_positions)] = pad_id
    return x


def make_kept_only_input(input_ids_1d, deletable_positions, keep_positions, pad_id):
    x = input_ids_1d.clone()
    keep_positions = set(int(p) for p in keep_positions)
    for p in deletable_positions:
        if int(p) not in keep_positions:
            x[int(p)] = pad_id
    return x


def compute_metrics_for_record(record, tokenizer, fractions, topk_frac=0.2, random_restarts=5, method_names=None):
    if method_names is None:
        method_names = CANDIDATE_NAMES

    pad_id = tokenizer.pad_token_id
    input_ids_1d = record["input_ids"].clone()
    pred_class = int(record["pred_label"])
    deletable = record["deletable_indices"]
    n_del = len(deletable)

    base_prob = predict_prob_for_class(model, input_ids_1d, pred_class, pad_id)
    out = {}

    for method in method_names:
        if method == "random":
            curve_accum, comp_accum, suff_accum = [], [], []
            for rr in range(random_restarts):
                rng = np.random.default_rng(
                    GLOBAL_SEED + rr + int(record["true_label"]) * 1000 + len(record["text"])
                )
                order = rank_positions(record, "random", rng=rng)
                curve = [base_prob]
                for frac in fractions:
                    k = max(1, int(math.ceil(frac * n_del)))
                    remove_pos = order[:k]
                    masked_ids = make_removed_input(input_ids_1d, remove_pos, pad_id)
                    p = predict_prob_for_class(model, masked_ids, pred_class, pad_id)
                    curve.append(p)

                k_top = max(1, int(math.ceil(topk_frac * n_del)))
                top_pos = order[:k_top]
                removed_ids = make_removed_input(input_ids_1d, top_pos, pad_id)
                p_removed = predict_prob_for_class(model, removed_ids, pred_class, pad_id)
                comp = base_prob - p_removed
                kept_ids = make_kept_only_input(input_ids_1d, deletable, top_pos, pad_id)
                p_kept = predict_prob_for_class(model, kept_ids, pred_class, pad_id)
                suff = base_prob - p_kept

                curve_accum.append(curve)
                comp_accum.append(comp)
                suff_accum.append(suff)

            curve = np.mean(np.array(curve_accum), axis=0)
            comp = float(np.mean(comp_accum))
            suff = float(np.mean(suff_accum))
        else:
            order = rank_positions(record, method_name=method)
            curve = [base_prob]
            for frac in fractions:
                k = max(1, int(math.ceil(frac * n_del)))
                remove_pos = order[:k]
                masked_ids = make_removed_input(input_ids_1d, remove_pos, pad_id)
                p = predict_prob_for_class(model, masked_ids, pred_class, pad_id)
                curve.append(p)

            k_top = max(1, int(math.ceil(topk_frac * n_del)))
            top_pos = order[:k_top]
            removed_ids = make_removed_input(input_ids_1d, top_pos, pad_id)
            p_removed = predict_prob_for_class(model, removed_ids, pred_class, pad_id)
            comp = base_prob - p_removed
            kept_ids = make_kept_only_input(input_ids_1d, deletable, top_pos, pad_id)
            p_kept = predict_prob_for_class(model, kept_ids, pred_class, pad_id)
            suff = base_prob - p_kept
            curve = np.array(curve, dtype=np.float64)

        x = np.concatenate([[0.0], fractions])
        auc = float(trapezoid(curve, x=x))
        out[method] = {
            "curve": np.asarray(curve, dtype=np.float64),
            "auc": auc,
            "comprehensiveness": float(comp),
            "sufficiency": float(suff),
        }
    return base_prob, out


def summarize_metrics_df(metrics_df, method_names):
    rows = []
    for method in method_names:
        auc_vals = metrics_df[f"{method}_auc"].values
        comp_vals = metrics_df[f"{method}_comp"].values
        suff_vals = metrics_df[f"{method}_suff"].values
        ci_low, ci_high = bootstrap_mean_ci(auc_vals, n_boot=BOOTSTRAP_B, seed=BOOTSTRAP_SEED)
        rows.append({
            "method": method,
            "auc_mean": float(np.mean(auc_vals)),
            "auc_std": float(np.std(auc_vals, ddof=1)),
            "auc_ci_low": ci_low,
            "auc_ci_high": ci_high,
            "comp_mean": float(np.mean(comp_vals)),
            "suff_mean": float(np.mean(suff_vals)),
        })
    return pd.DataFrame(rows).sort_values(["auc_mean", "comp_mean"], ascending=[True, False]).reset_index(drop=True)


def repeated_stratified_auc_runs(metrics_df, labels_arr, method_names, n_runs=10, run_sample_size=300):
    rows = []
    labels_arr = np.asarray(labels_arr)

    for run_seed in range(n_runs):
        idx = stratified_subsample_indices(labels_arr, run_sample_size, seed=1000 + run_seed)
        sub = metrics_df.iloc[idx]

        for method in method_names:
            rows.append({
                "run_seed": 1000 + run_seed,
                "method": method,
                "mean_auc": float(sub[f"{method}_auc"].mean()),
            })

    runs_df = pd.DataFrame(rows)

    agg = (
        runs_df.groupby("method", as_index=False)
        .agg(
            repeated_mean_auc=("mean_auc", "mean"),
            repeated_std_auc=("mean_auc", "std"),
        )
        .sort_values("repeated_mean_auc")
        .reset_index(drop=True)
    )

    return runs_df, agg

def pairwise_tests(metrics_df, method_names):
    pairs = []
    for metric_key, suffix in [("auc", "auc"), ("comprehensiveness", "comp"), ("sufficiency", "suff")]:
        for i in range(len(method_names)):
            for j in range(i + 1, len(method_names)):
                a = method_names[i]
                b = method_names[j]
                xa = metrics_df[f"{a}_{suffix}"].values
                xb = metrics_df[f"{b}_{suffix}"].values
                stat, p = wilcoxon(xa, xb, zero_method="wilcox", alternative="two-sided")
                pairs.append({
                    "metric": metric_key,
                    "method_a": a,
                    "method_b": b,
                    "mean_diff_a_minus_b": float(np.mean(xa - xb)),
                    "statistic": float(stat),
                    "p_value": float(p),
                })
    pairwise_df = pd.DataFrame(pairs)
    pairwise_out = []
    for metric in pairwise_df["metric"].unique():
        pairwise_out.append(holm_bonferroni(pairwise_df[pairwise_df["metric"] == metric]))
    return pd.concat(pairwise_out, axis=0).reset_index(drop=True)

# ============================================================
# RUN EXPLANATIONS (SCREENING CACHE)
# ============================================================

print(">>> Building explanation cache for screening...")
records = []
for i, row in test_df.iterrows():
    if i % 25 == 0:
        print(f"    processed {i}/{len(test_df)}")
    rec = explain_instance(
        model=model,
        tokenizer=tokenizer,
        text=row["text"],
        true_label=row["label"],
        m_steps=M_STEPS_SCREEN,
        max_length=MAX_LENGTH,
    )
    records.append(rec)
print(">>> Explanation cache ready.")

# ============================================================
# SCREENING ROUND A/B
# ============================================================

print(">>> Screening candidate methods...")
metric_rows = []
curve_store = defaultdict(list)
for rec in records:
    base_prob, per_method = compute_metrics_for_record(
        rec,
        tokenizer,
        FRACTIONS,
        topk_frac=TOPK_FRAC_COMP,
        random_restarts=RANDOM_RESTARTS,
        method_names=CANDIDATE_NAMES,
    )

    row = {
        "true_label": rec["true_label"],
        "pred_label": rec["pred_label"],
        "pred_conf": rec["pred_conf"],
        "num_deletable": rec["num_deletable"],
    }

    default_bundle = build_score_bundle(record=rec, norm="sum", alpha_fixed=0.85, alpha_min=0.65, alpha_max=0.95)
    row["spearman_attn_ig"] = default_bundle["spearman_attn_ig"]
    row["alpha_adaptive_default"] = default_bundle["alpha_adaptive"]

    for method_name, met in per_method.items():
        row[f"{method_name}_auc"] = met["auc"]
        row[f"{method_name}_comp"] = met["comprehensiveness"]
        row[f"{method_name}_suff"] = met["sufficiency"]
        curve_store[method_name].append(met["curve"])

    metric_rows.append(row)

metrics_df = pd.DataFrame(metric_rows)
metrics_df.to_csv(os.path.join(OUT_DIR, "screening_metrics_all_candidates.csv"), index=False)

summary_df = summarize_metrics_df(metrics_df, CANDIDATE_NAMES)
summary_df.to_csv(os.path.join(OUT_DIR, "screening_summary_all_candidates.csv"), index=False)

hybrid_fixed_summary = summary_df[summary_df["method"].str.startswith("hybrid_fixed")].reset_index(drop=True)
hybrid_adaptive_summary = summary_df[summary_df["method"].str.startswith("hybrid_adaptive")].reset_index(drop=True)

best_fixed_method = hybrid_fixed_summary.iloc[0]["method"]
best_adaptive_method = hybrid_adaptive_summary.iloc[0]["method"]

print(f">>> Best fixed hybrid from screening: {best_fixed_method}")
print(f">>> Best adaptive hybrid from screening: {best_adaptive_method}")

# ============================================================
# DEEP ROUND C: REBUILD CACHE WITH MORE IG STEPS
# ============================================================

print(">>> Rebuilding explanation cache for deep evaluation...")
records_deep = []
for i, row in test_df.iterrows():
    if i % 25 == 0:
        print(f"    deep cache {i}/{len(test_df)}")
    rec = explain_instance(
        model=model,
        tokenizer=tokenizer,
        text=row["text"],
        true_label=row["label"],
        m_steps=M_STEPS_DEEP,
        max_length=MAX_LENGTH,
    )
    records_deep.append(rec)
print(">>> Deep explanation cache ready.")

FINAL_METHODS = ["attention", "ig", best_fixed_method, best_adaptive_method, "random"]
print(">>> Final shortlist:", FINAL_METHODS)

metric_rows = []
curve_store = defaultdict(list)
for rec in records_deep:
    base_prob, per_method = compute_metrics_for_record(
        rec,
        tokenizer,
        FRACTIONS,
        topk_frac=TOPK_FRAC_COMP,
        random_restarts=RANDOM_RESTARTS,
        method_names=FINAL_METHODS,
    )

    fixed_bundle = build_score_bundle(rec, **{
        "norm": parse_method_tag(best_fixed_method)["norm"],
        "alpha_fixed": parse_method_tag(best_fixed_method)["alpha"],
        "alpha_min": 0.65,
        "alpha_max": 0.95,
    })
    adaptive_bundle = build_score_bundle(rec, **{
        "norm": parse_method_tag(best_adaptive_method)["norm"],
        "alpha_fixed": 0.85,
        "alpha_min": parse_method_tag(best_adaptive_method)["amin"],
        "alpha_max": parse_method_tag(best_adaptive_method)["amax"],
    })

    row = {
        "text": rec["text"],
        "true_label": rec["true_label"],
        "pred_label": rec["pred_label"],
        "pred_conf": rec["pred_conf"],
        "num_deletable": rec["num_deletable"],
        "spearman_attn_ig": fixed_bundle["spearman_attn_ig"],
        "alpha_adaptive": adaptive_bundle["alpha_adaptive"],
    }

    for method_name, met in per_method.items():
        row[f"{method_name}_auc"] = met["auc"]
        row[f"{method_name}_comp"] = met["comprehensiveness"]
        row[f"{method_name}_suff"] = met["sufficiency"]
        curve_store[method_name].append(met["curve"])
    metric_rows.append(row)

metrics_df = pd.DataFrame(metric_rows)
metrics_df.to_csv(os.path.join(OUT_DIR, "final_metrics_shortlist.csv"), index=False)

summary_df = summarize_metrics_df(metrics_df, FINAL_METHODS)
summary_df.to_csv(os.path.join(OUT_DIR, "summary_overall.csv"), index=False)

runs_df, runs_agg = repeated_stratified_auc_runs(
    metrics_df,
    metrics_df["true_label"].values,
    FINAL_METHODS,
    n_runs=N_RUNS,
    run_sample_size=RUN_SAMPLE_SIZE,
)
runs_df.to_csv(os.path.join(OUT_DIR, "repeated_runs_aggregate.csv"), index=False)
runs_agg.to_csv(os.path.join(OUT_DIR, "repeated_runs_summary.csv"), index=False)

friedman_stat, friedman_p = friedmanchisquare(*[metrics_df[f"{m}_auc"].values for m in FINAL_METHODS])
friedman_df = pd.DataFrame([{"statistic": friedman_stat, "p_value": friedman_p}])
friedman_df.to_csv(os.path.join(OUT_DIR, "friedman_auc.csv"), index=False)

pairwise_df = pairwise_tests(metrics_df, FINAL_METHODS)
pairwise_df.to_csv(os.path.join(OUT_DIR, "pairwise_wilcoxon_with_holm.csv"), index=False)

# ============================================================
# TOP-K SWEEP FOR FINAL SHORTLIST
# ============================================================

print(">>> Running top-k sweep for final shortlist...")
topk_rows = []
for topk_frac in TOPK_FRAC_SWEEP:
    temp_rows = []
    for rec in records_deep:
        _, per_method = compute_metrics_for_record(
            rec,
            tokenizer,
            FRACTIONS,
            topk_frac=topk_frac,
            random_restarts=RANDOM_RESTARTS,
            method_names=FINAL_METHODS,
        )
        row = {"true_label": rec["true_label"], "topk_frac": topk_frac}
        for method_name, met in per_method.items():
            row[f"{method_name}_comp"] = met["comprehensiveness"]
            row[f"{method_name}_suff"] = met["sufficiency"]
        temp_rows.append(row)
    temp_df = pd.DataFrame(temp_rows)
    for method in FINAL_METHODS:
        topk_rows.append({
            "topk_frac": topk_frac,
            "method": method,
            "comp_mean": float(temp_df[f"{method}_comp"].mean()),
            "suff_mean": float(temp_df[f"{method}_suff"].mean()),
        })

topk_df = pd.DataFrame(topk_rows)
topk_df.to_csv(os.path.join(OUT_DIR, "topk_sweep_summary.csv"), index=False)

# ============================================================
# CLASS-WISE SUMMARY FOR FINAL SHORTLIST
# ============================================================

classwise_rows = []
for cls in sorted(metrics_df["true_label"].unique()):
    sub = metrics_df[metrics_df["true_label"] == cls]
    for method in FINAL_METHODS:
        vals = sub[f"{method}_auc"].values
        classwise_rows.append({
            "true_label": cls,
            "method": method,
            "mean_auc": float(np.mean(vals)),
            "std_auc": float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0,
            "n": len(vals),
        })

classwise_df = pd.DataFrame(classwise_rows)
classwise_df.to_csv(os.path.join(OUT_DIR, "classwise_auc.csv"), index=False)
classwise_pivot = classwise_df.pivot(index="true_label", columns="method", values="mean_auc")
classwise_pivot.to_csv(os.path.join(OUT_DIR, "classwise_auc_pivot.csv"))

# ============================================================
# HARD-CASE ANALYSIS
# ============================================================

median_conf = metrics_df["pred_conf"].median()
median_len = metrics_df["num_deletable"].median()
subset_specs = [
    ("low_confidence", metrics_df["pred_conf"] <= median_conf),
    ("high_confidence", metrics_df["pred_conf"] > median_conf),
    ("short_text", metrics_df["num_deletable"] <= median_len),
    ("long_text", metrics_df["num_deletable"] > median_len),
]

subset_rows = []
for subset_name, mask in subset_specs:
    sub = metrics_df[mask]
    for method in FINAL_METHODS:
        subset_rows.append({
            "subset": subset_name,
            "method": method,
            "mean_auc": float(sub[f"{method}_auc"].mean()),
            "mean_comp": float(sub[f"{method}_comp"].mean()),
            "mean_suff": float(sub[f"{method}_suff"].mean()),
            "n": int(len(sub)),
        })
subset_df = pd.DataFrame(subset_rows)
subset_df.to_csv(os.path.join(OUT_DIR, "subset_analysis.csv"), index=False)

# ============================================================
# PLOTS
# ============================================================

print(">>> Saving publication-ready plots...")

plt.rcParams.update({
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "font.size": 13,
    "axes.titlesize": 16,
    "axes.labelsize": 15,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 11,
    "legend.frameon": True,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

plot_method_labels = {
    "attention": "Attention",
    "ig": "Integrated Gradients",
    best_fixed_method: "Hybrid (Fixed α)",
    best_adaptive_method: "Hybrid (Adaptive α)",
    "random": "Random",
}

plot_colors = {
    "attention": "#1f77b4",
    "ig": "#ff7f0e",
    best_fixed_method: "#2ca02c",
    best_adaptive_method: "#d62728",
    "random": "#7f7f7f",
}

def apply_axis_style(ax, grid_axis="y"):
    ax.grid(True, axis=grid_axis, linestyle="--", linewidth=0.7, alpha=0.35)
    ax.set_axisbelow(True)

def styled_boxplot(ax, data, labels, ylabel, title, colors):
    bp = ax.boxplot(
        data,
        labels=labels,
        showmeans=True,
        patch_artist=True,
        medianprops={"color": "black", "linewidth": 1.6},
        whiskerprops={"linewidth": 1.2},
        capprops={"linewidth": 1.2},
        boxprops={"linewidth": 1.2},
        meanprops={
            "marker": "o",
            "markerfacecolor": "white",
            "markeredgecolor": "black",
            "markersize": 6,
        },
        flierprops={
            "marker": "o",
            "markerfacecolor": "gray",
            "markeredgecolor": "gray",
            "markersize": 3,
            "alpha": 0.45,
        },
    )
    for patch, color in zip(bp["boxes"], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.55)
    ax.set_ylabel(ylabel)
    ax.set_title(title, pad=10)
    apply_axis_style(ax, grid_axis="y")

def save_current_figure(path):
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    plt.close()

x = np.concatenate([[0.0], FRACTIONS])

fig, ax = plt.subplots(figsize=(7.6, 5.6), dpi=DPI)
for method in FINAL_METHODS:
    curves = np.array(curve_store[method])
    mean_curve = curves.mean(axis=0)
    sem_curve = curves.std(axis=0, ddof=1) / np.sqrt(len(curves))
    lo = mean_curve - 1.96 * sem_curve
    hi = mean_curve + 1.96 * sem_curve
    ax.plot(
        x, mean_curve, marker="o", markersize=5, linewidth=2.2,
        color=plot_colors[method], label=plot_method_labels[method]
    )
    ax.fill_between(x, lo, hi, color=plot_colors[method], alpha=0.14)
ax.set_xlabel("Fraction of Tokens Removed")
ax.set_ylabel("Predicted Probability of Original Class")
ax.set_title("Deletion Curves (MoRF) with 95% Confidence Intervals", pad=10)
ax.set_xlim(0.0, 1.0)
apply_axis_style(ax, grid_axis="both")
ax.legend(loc="best", ncol=1)
save_current_figure(os.path.join(OUT_DIR, "fig01_deletion_curves_morf_ci_pubready.png"))

fig, ax = plt.subplots(figsize=(8.8, 5.8), dpi=DPI)
box_data = [metrics_df[f"{m}_auc"].values for m in FINAL_METHODS]
styled_boxplot(
    ax,
    box_data,
    [plot_method_labels[m] for m in FINAL_METHODS],
    "Deletion AUC (Lower is Better)",
    "Per-Instance Deletion AUC Distribution",
    [plot_colors[m] for m in FINAL_METHODS],
)
ax.tick_params(axis="x", rotation=15)
save_current_figure(os.path.join(OUT_DIR, "fig02_deletion_auc_boxplot_pubready.png"))

fig, ax = plt.subplots(figsize=(8.8, 5.8), dpi=DPI)
box_data = [metrics_df[f"{m}_comp"].values for m in FINAL_METHODS]
styled_boxplot(
    ax,
    box_data,
    [plot_method_labels[m] for m in FINAL_METHODS],
    f"Comprehensiveness at Top {int(TOPK_FRAC_COMP * 100)}% (Higher is Better)",
    "Per-Instance Comprehensiveness Distribution",
    [plot_colors[m] for m in FINAL_METHODS],
)
ax.tick_params(axis="x", rotation=15)
save_current_figure(os.path.join(OUT_DIR, "fig03_comprehensiveness_boxplot_pubready.png"))

fig, ax = plt.subplots(figsize=(8.8, 5.8), dpi=DPI)
box_data = [metrics_df[f"{m}_suff"].values for m in FINAL_METHODS]
styled_boxplot(
    ax,
    box_data,
    [plot_method_labels[m] for m in FINAL_METHODS],
    f"Sufficiency Gap at Top {int(TOPK_FRAC_COMP * 100)}% (Lower is Better)",
    "Per-Instance Sufficiency Distribution",
    [plot_colors[m] for m in FINAL_METHODS],
)
ax.tick_params(axis="x", rotation=15)
save_current_figure(os.path.join(OUT_DIR, "fig04_sufficiency_boxplot_pubready.png"))

fig, ax = plt.subplots(figsize=(7.2, 5.4), dpi=DPI)
vals = metrics_df["spearman_attn_ig"].values
ax.hist(vals, bins=20, color="#4c78a8", alpha=0.82, edgecolor="white")
ax.axvline(vals.mean(), color="black", linestyle="--", linewidth=1.8, label=f"Mean = {vals.mean():.3f}")
ax.axvline(np.median(vals), color="black", linestyle=":", linewidth=1.8, label=f"Median = {np.median(vals):.3f}")
ax.set_xlabel("Spearman Correlation Between Attention and IG")
ax.set_ylabel("Number of Instances")
ax.set_title("Attention–IG Rank Correlation Distribution", pad=10)
apply_axis_style(ax, grid_axis="y")
ax.legend(loc="best")
save_current_figure(os.path.join(OUT_DIR, "fig05_attention_ig_spearman_histogram_pubready.png"))

fig, ax = plt.subplots(figsize=(7.2, 5.4), dpi=DPI)
alpha_vals = metrics_df["alpha_adaptive"].values
ax.hist(alpha_vals, bins=20, color="#e45756", alpha=0.82, edgecolor="white")
ax.axvline(alpha_vals.mean(), color="black", linestyle="--", linewidth=1.8, label=f"Mean = {alpha_vals.mean():.3f}")
ax.axvline(np.median(alpha_vals), color="black", linestyle=":", linewidth=1.8, label=f"Median = {np.median(alpha_vals):.3f}")
ax.set_xlabel("Adaptive α")
ax.set_ylabel("Number of Instances")
ax.set_title("Adaptive Weight Distribution", pad=10)
apply_axis_style(ax, grid_axis="y")
ax.legend(loc="best")
save_current_figure(os.path.join(OUT_DIR, "fig06_adaptive_alpha_histogram_pubready.png"))

fig, ax = plt.subplots(figsize=(8.2, 4.8), dpi=DPI)
mat = classwise_pivot.values
im = ax.imshow(mat, aspect="auto", cmap="viridis")
cbar = fig.colorbar(im, ax=ax)
cbar.set_label("Mean Deletion AUC")
ax.set_xticks(range(classwise_pivot.shape[1]))
ax.set_xticklabels([plot_method_labels.get(c, c) for c in classwise_pivot.columns], rotation=20, ha="right")
ax.set_yticks(range(classwise_pivot.shape[0]))
ax.set_yticklabels(classwise_pivot.index)
ax.set_xlabel("Explanation Method")
ax.set_ylabel("True Label")
ax.set_title("Class-Wise Mean Deletion AUC Heatmap", pad=10)
save_current_figure(os.path.join(OUT_DIR, "fig07_classwise_deletion_auc_heatmap_pubready.png"))

fig, ax = plt.subplots(figsize=(7.6, 5.6), dpi=DPI)
for method in FINAL_METHODS:
    sub = topk_df[topk_df["method"] == method]
    ax.plot(
        sub["topk_frac"], sub["comp_mean"], marker="o", markersize=5, linewidth=2.0,
        color=plot_colors[method], label=plot_method_labels[method]
    )
ax.set_xlabel("Top-k Fraction")
ax.set_ylabel("Mean Comprehensiveness")
ax.set_title("Top-k Sensitivity of Comprehensiveness", pad=10)
apply_axis_style(ax, grid_axis="both")
ax.legend(loc="best")
save_current_figure(os.path.join(OUT_DIR, "fig08_topk_comprehensiveness_sweep_pubready.png"))

fig, ax = plt.subplots(figsize=(7.6, 5.6), dpi=DPI)
for method in FINAL_METHODS:
    sub = topk_df[topk_df["method"] == method]
    ax.plot(
        sub["topk_frac"], sub["suff_mean"], marker="o", markersize=5, linewidth=2.0,
        color=plot_colors[method], label=plot_method_labels[method]
    )
ax.set_xlabel("Top-k Fraction")
ax.set_ylabel("Mean Sufficiency Gap")
ax.set_title("Top-k Sensitivity of Sufficiency", pad=10)
apply_axis_style(ax, grid_axis="both")
ax.legend(loc="best")
save_current_figure(os.path.join(OUT_DIR, "fig09_topk_sufficiency_sweep_pubready.png"))

# ============================================================
# TOKEN EXPORTS FOR FINAL SHORTLIST
# ============================================================

def top_tokens_from_record(rec, method, topn=10):
    scores, bundle = get_scores_for_method(rec, method)
    if method == "attention":
        alpha_adaptive = np.nan
    elif method == "ig":
        alpha_adaptive = np.nan
    elif method.startswith("hybrid_adaptive"):
        alpha_adaptive = bundle["alpha_adaptive"]
    else:
        alpha_adaptive = np.nan

    idx = rec["deletable_indices"]
    order = idx[np.argsort(-scores[idx])][:topn]
    pairs = [(rec["tokens_full"][i], float(scores[i])) for i in order]
    return pairs, alpha_adaptive


token_rows = []
for i, rec in enumerate(records_deep):
    for method in ["attention", "ig", best_fixed_method, best_adaptive_method]:
        pairs, alpha_adaptive = top_tokens_from_record(rec, method, topn=10)
        token_rows.append({
            "id": i,
            "true_label": rec["true_label"],
            "pred_label": rec["pred_label"],
            "method": plot_method_labels.get(method, method),
            "method_full": method,
            "alpha_adaptive": alpha_adaptive,
            "top_tokens": json.dumps(pairs, ensure_ascii=False),
            "text": rec["text"],
        })

top_tokens_df = pd.DataFrame(token_rows)
top_tokens_df.to_csv(os.path.join(OUT_DIR, "top_tokens_per_instance.csv"), index=False)

# ============================================================
# FINAL REPORT
# ============================================================

best_auc_method = summary_df.iloc[0]["method"]
report_lines = []
report_lines.append("Explainability / Faithfulness Summary")
report_lines.append("=" * 60)
report_lines.append(f"Instances analyzed: {len(metrics_df)}")
report_lines.append(f"Device: {device}")
report_lines.append(f"IG steps screening: {M_STEPS_SCREEN}")
report_lines.append(f"IG steps deep evaluation: {M_STEPS_DEEP}")
report_lines.append(f"Deletion fractions: {FRACTIONS.tolist()}")
report_lines.append(f"Top-k default for comp/suff: {TOPK_FRAC_COMP}")
report_lines.append(f"Top-k sweep: {TOPK_FRAC_SWEEP}")
report_lines.append("")
report_lines.append("Best screening methods:")
report_lines.append(f"  best fixed hybrid    : {best_fixed_method}")
report_lines.append(f"  best adaptive hybrid : {best_adaptive_method}")
report_lines.append("")
report_lines.append("Overall ranking by deletion AUC (lower is better):")
for _, r in summary_df.iterrows():
    report_lines.append(
        f"  {plot_method_labels.get(r['method'], r['method']):>16s} | "
        f"AUC {r['auc_mean']:.4f} ± {r['auc_std']:.4f} "
        f"[95% CI {r['auc_ci_low']:.4f}, {r['auc_ci_high']:.4f}] | "
        f"Comp {r['comp_mean']:.4f} | Suff {r['suff_mean']:.4f}"
    )
report_lines.append("")
report_lines.append("Repeated stratified runs (mean of run-level means):")
for _, r in runs_agg.iterrows():
    report_lines.append(
        f"  {plot_method_labels.get(r['method'], r['method']):>16s} | "
        f"mean AUC {r['repeated_mean_auc']:.4f} ± {r['repeated_std_auc']:.4f}"
    )
report_lines.append("")
report_lines.append(
    f"Friedman test on per-instance AUCs: stat={friedman_stat:.4f}, p={friedman_p:.6g}"
)
report_lines.append("")
report_lines.append("Pairwise Wilcoxon tests (Holm-adjusted):")
for _, r in pairwise_df.sort_values(["metric", "p_holm"]).iterrows():
    report_lines.append(
        f"  {r['metric']:>18s} | {plot_method_labels.get(r['method_a'], r['method_a'])} "
        f"vs {plot_method_labels.get(r['method_b'], r['method_b'])} | "
        f"mean diff={r['mean_diff_a_minus_b']:.6f} | p={r['p_value']:.6g} | p_holm={r['p_holm']:.6g}"
    )
report_lines.append("")
report_lines.append(
    f"Attention–IG Spearman: mean={metrics_df['spearman_attn_ig'].mean():.4f}, "
    f"median={metrics_df['spearman_attn_ig'].median():.4f}"
)
report_lines.append(
    f"Adaptive alpha (best adaptive hybrid): mean={metrics_df['alpha_adaptive'].mean():.4f}, "
    f"median={metrics_df['alpha_adaptive'].median():.4f}, "
    f"min={metrics_df['alpha_adaptive'].min():.4f}, "
    f"max={metrics_df['alpha_adaptive'].max():.4f}"
)
report_lines.append("")
report_lines.append(f"Best AUC method: {plot_method_labels.get(best_auc_method, best_auc_method)}")

with open(os.path.join(OUT_DIR, "report.txt"), "w", encoding="utf-8") as f:
    f.write("\n".join(report_lines))

print("\n".join(report_lines))
print(f"\n>>> Done. All outputs saved to: {OUT_DIR}")


/home/btlab/anaconda3/envs/llm-misinfo/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


>>> Using device: cuda
>>> Candidate methods in screening stage: 43
>>> Loading data...
>>> Test instances available for XAI analysis: 600
>>> Loading model...


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 3076.10it/s]
RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


>>> Missing keys: []
>>> Unexpected keys: []
>>> Model loaded successfully.
>>> Building explanation cache for screening...
    processed 0/600
    processed 25/600
    processed 50/600
    processed 75/600
    processed 100/600
    processed 125/600
    processed 150/600
    processed 175/600
    processed 200/600
    processed 225/600
    processed 250/600
    processed 275/600
    processed 300/600
    processed 325/600
    processed 350/600
    processed 375/600
    processed 400/600
    processed 425/600
    processed 450/600
    processed 475/600
    processed 500/600
    processed 525/600
    processed 550/600
    processed 575/600
>>> Explanation cache ready.
>>> Screening candidate methods...
>>> Best fixed hybrid from screening: hybrid_fixed__norm=minmax__alpha=0.60
>>> Best adaptive hybrid from screening: hybrid_adaptive__norm=sum__amin=0.10__amax=0.60
>>> Rebuilding explanation cache for deep evaluation...
    deep cache 0/600
    deep cache 25/600
    deep cache 50/600
   